# x4a_pecuaria — Lotação Aparente de Pastagens (Cabeças/ha de Pastagem)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** lotação aparente de pastagens, cabeças/ha de pastagem, municipal, anual*  
**Conceito e Mecanismo Econômico:** Intensidade da pecuária, calculada pela divisão entre o efetivo bovino (PPM) e a área de pastagem (MapBiomas classe 15).  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1_h2__ppm_bovinos__ibge__municipio_anual.csv`
- `2_INTI\xdata\chap1\processed\tabular\h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Candidato a construir: efetivo bovino / área de pastagem (classe 15), com geografia e datas compatíveis.
> - Avaliar denominadores pequenos/nulos; não equivale a rendimento pecuário ou rentabilidade.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano.
- **`efetivo_bovino`**: Rebanho bovino.
- **`area_pastagem_ha`**: Área de pastagem do MapBiomas.
- **`lotacao_cabecas_ha`**: Taxa de lotação aparente (cabeças/ha).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1_h2__ppm_bovinos__ibge__municipio_anual.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x4a_pecuaria`:
- **Atenção:** Candidato a construir: efetivo bovino / área de pastagem (classe 15), com geografia e datas compatíveis.
- **Atenção:** Avaliar denominadores pequenos/nulos; não equivale a rendimento pecuário ou rentabilidade.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
# REGRA DO TXT:
# 'Candidato a construir: efetivo bovino / área de pastagem (classe 15), com geografia e datas compatíveis.'
# 'Avaliar denominadores pequenos/nulos; não equivale a rendimento pecuário ou rentabilidade.'
# Harmonização territorial para 141 municípios aplicada em ambas as bases.

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Verificação de salvaguarda: denominadores de pastagem < 100 ha geram distorções de lotação infinita.")

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
print("Análise univariada da taxa de lotação...")

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
print("Evolução da intensidade pecuária em MT (2001-2024)...")

## Fase 6: Feature Engineering — Lotação Pecuária Aparente (Cabeças/ha)

### 1. Step 1: Junção de Rebanho (PPM) e Área de Pastagem (MapBiomas)
- **Fórmula da Densidade de Pastejo:**
  $$Lotacao_{i, t} = \frac{\text{Efetivo Bovino (cabeças)}_{i, t}}{\text{Área de Pastagem (ha)}_{i, t}}$$

### 2. Step 2: Salvaguardas Matemáticas e Tratamento de Outliers
- **Salvaguarda de Denominador Pequeno:** Se a área de pastagem for menor que 200 hectares (`area_pastagem_ha < 200`), a lotação é definida como `NaN`. Em cidades quase 100% agrícolas, alguns bois de subsistência em piquetes de 20 ha gerariam taxas absurdas de 40 cabeças/ha!
- **Winsorização:** Truncamos a lotação ao teto biológico de **4,0 cabeças/ha** (ou percentil 99), para eliminar distorções de confinamentos intensivos pontuais que não representam o uso extensivo da terra.
- **Defasagem $t-1$ e Variação ($\Delta Lotação$):** Aumentos na lotação indicam **intensificação pecuária** (poupança de terra / land-sparing), diminuindo a demanda por desmatamento.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Cálculo da Lotação Bruta (Cabeças de Gado por Hectare de Pasto)
df_lot = df.copy()

# Salvaguarda: pastagem mínima de 200 ha para evitar distorção matemática de divisão
PASTAGEM_MINIMA_HA = 200.0

df_lot['lotacao_aparente'] = np.where(
    df_lot['area_pastagem_ha'] >= PASTAGEM_MINIMA_HA,
    df_lot['efetivo_bovino'] / df_lot['area_pastagem_ha'],
    np.nan
)

# 2. Winsorização de Outliers Extremos (Teto biológico de pastejo extensivo = 4.0 cabeças/ha)
df_lot['lotacao_winsorizada'] = df_lot['lotacao_aparente'].clip(lower=0.0, upper=4.0)

# 3. Defasagem Temporal de 1 ano (t-1)
df_lot.sort_values(['codigo_ibge', 'ano'], inplace=True)
df_lot['lotacao_lag1'] = df_lot.groupby('codigo_ibge')['lotacao_winsorizada'].shift(1)

# 4. Indicador de Intensificação Pecuária (Delta Lotação)
# delta_lotacao > 0: intensificação (mais bois por hectare, poupando floresta)
# delta_lotacao < 0: degradação de pastagens ou extensificação
df_lot['delta_lotacao'] = df_lot['lotacao_winsorizada'] - df_lot['lotacao_lag1']

painel_lot = df_lot[(df_lot['ano'] >= 2001) & (df_lot['ano'] <= 2024)].copy()

print("Resumo da Lotação Pecuária Aparente (Cabeças/ha):")
print(painel_lot[['lotacao_aparente', 'lotacao_winsorizada', 'lotacao_lag1', 'delta_lotacao']].describe())
print(painel_lot[['codigo_ibge', 'ano', 'efetivo_bovino', 'area_pastagem_ha', 'lotacao_winsorizada', 'lotacao_lag1']].head())